In [ ]:
import json
from groq import Groq
from config import *
from data_loader import load_legal_context
from utils import parse_json_response

In [ ]:
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
SCENARIO_VARIETY_INSTRUCTIONS = """SCENARIO VARIETY INSTRUCTIONS:
- Before choosing any fictional proper noun, internally generate at least 10 distinct options and select one. Do not reveal the candidate list.
- Apply this to names, addresses, streets, neighborhoods, businesses, schools, hospitals, agencies, parks, apartment complexes, workplaces, and any other fictional proper noun.
- Avoid generic placeholder names and overly repetitive combinations.
- Avoid similar-sounding names within the same scenario, such as two people with similar first names or the same last name unless they are related.
- Vary the setting, time of day, reporting path, relationship pattern, household context, and incident structure.
- Vary victim, caller, suspect, and witness demographics across generated scenarios.
- Do not default to a woman in her 20s as the victim, caller, or primary reporting party.
- Internally consider multiple age ranges, genders, household types, relationship dynamics, living situations, and reporting-party roles before selecting the final scenario setup.
- Keep demographic details realistic and relevant to the scenario, but do not make demographic traits the focus unless directly relevant to the facts.
- Use only the final selected names, places, demographics, and scenario details in the JSON output.
- Do not mention prior scenarios or attempt to compare against prior generated examples."""

DEFAULT_CRIME_TYPE_CONSTRAINTS = """CRIME-TYPE CONSTRAINTS:
- Keep the scenario focused on the selected primary crime type.
- Do not add unrelated co-occurring crimes unless a checklist element directly requires the topic.
- If a checklist element asks about conduct that did not occur, provide a concrete negative fact instead of inventing that conduct.
- The facts should support the selected primary crime type rather than turning the scenario into a different type of incident.
- Do not make the scenario more severe or complex than needed to satisfy the checklist."""

CRIME_TYPE_CONSTRAINTS = {
    "domestic violence": """CRIME-TYPE CONSTRAINTS:
- The primary incident must be domestic violence.
- Focus the scenario on the relationship between the parties, the reported domestic incident, prior history if relevant, victim safety, injuries if present, threats if present, children if present, witnesses, evidence, and officer response.
- Do not automatically add stalking, strangulation, sexual assault, firearm prohibition, active warrants, or serious felony-level facts unless the checklist directly requires that topic.
- If a checklist element asks about related conduct that did not occur, provide a concrete negative fact instead of inventing that conduct.
- Keep any related protection-order, firearm, stalking, strangulation, medical, or sexual-violence facts limited and consistent with the domestic violence incident.""",

    "stalking": """CRIME-TYPE CONSTRAINTS:
- The primary incident must be stalking.
- Focus the scenario on repeated unwanted contact, following, surveillance, cyberstalking, unwanted gifts/messages, threats, fear, safety planning, documentation, and evidence of a course of conduct.
- Do not turn the scenario into a strangulation, sexual assault, serious physical assault, firearm-prohibition, or active-warrant scenario.
- If a checklist element asks about strangulation, sexual violence, serious assault, firearms, protection orders, or warrants, answer that element with a concrete negative or limited procedural fact unless that fact is strictly necessary and consistent with stalking as the primary crime.
- Related domestic violence facts may appear only if they support the stalking pattern and do not become the main incident.
- Do not add unrelated co-occurring crimes just to make the scenario more dramatic.""",

    "sexual assault": """CRIME-TYPE CONSTRAINTS:
- The primary incident must be sexual assault.
- Focus the scenario on disclosure, consent-related facts, suspect/victim relationship if known, timeline, location, medical/SANE response if applicable, evidence preservation, witnesses if present, victim services, and officer response.
- Do not add unrelated domestic violence, stalking, strangulation, firearm, burglary, active-warrant, or protection-order facts unless the checklist directly requires that topic.
- If a checklist element asks about unrelated conduct that did not occur, provide a concrete negative fact instead of inventing that conduct.
- Keep the scenario trauma-informed and avoid unnecessary graphic detail.
- Do not turn the scenario into a domestic violence, stalking, or strangulation scenario unless those facts are directly required and remain secondary.""",

    "partner/family member assault": """CRIME-TYPE CONSTRAINTS:
- The primary incident must be partner or family member assault.
- Focus the scenario on the assaultive conduct, relationship between the parties, injuries if present, threats if present, prior related incidents if relevant, witnesses, scene observations, evidence, victim safety, and officer response.
- Do not automatically add stalking, strangulation, sexual assault, firearm prohibition, active warrants, or protection-order violations unless the checklist directly requires that topic.
- If a related issue did not occur, state a concrete negative fact instead of inventing it.
- Keep the scenario centered on the assault and the qualifying partner/family relationship.""",

    "protection order violation": """CRIME-TYPE CONSTRAINTS:
- The primary incident must be a protection order violation.
- Focus the scenario on the existence of the order, whether it was valid and active, service or notice to the restrained party, the specific prohibited contact or conduct, dates/times, location, witnesses, evidence, victim safety, and officer response.
- Do not automatically add new domestic violence assault, stalking, strangulation, sexual assault, firearm possession, or active warrants unless the checklist directly requires that topic.
- If the violation involved contact only, keep it as a contact/order violation and do not escalate it into a separate assault scenario.
- If a checklist element asks about related conduct that did not occur, provide a concrete negative fact instead of inventing that conduct.""",

    "strangulation": """CRIME-TYPE CONSTRAINTS:
- The primary incident must be strangulation.
- Focus the scenario on the mechanism of strangulation, breathing impairment, symptoms, visible and non-visible injuries, medical evaluation, suspect/victim relationship, threats if present, witnesses if present, evidence, and officer response.
- Do not turn the scenario into a sexual assault, stalking, firearm-prohibition, active-warrant, protection-order, or unrelated serious assault scenario unless the checklist directly requires that topic.
- If related domestic violence facts are present, keep them limited and consistent with the strangulation incident.
- If a checklist element asks about unrelated conduct that did not occur, provide a concrete negative fact instead of inventing it."""
}

In [ ]:
def get_crime_type_constraints(crime_type):
    normalized = crime_type.strip().lower()
    return CRIME_TYPE_CONSTRAINTS.get(normalized, DEFAULT_CRIME_TYPE_CONSTRAINTS)

In [ ]:
# Generate scenario

def generate_scenario(checklist):
    crime_type = checklist["crime_type"]
    legal_context = load_legal_context()
    crime_type_constraints = get_crime_type_constraints(crime_type)
    elements = checklist["required_elements"]

    elements_text_full = "\n".join([
        f'  {el["id"]}: {el["element"]}'
        for el in elements
    ])

    # Generate dispatch line and interviewee info
    meta_prompt = f"""You are creating a fictional training scenario for a police report writing simulator.

Crime type: {crime_type}

{crime_type_constraints}

{SCENARIO_VARIETY_INSTRUCTIONS}

NAMES AND ADDRESSES:
{NAMES_AND_ADDRESSES}

Generate the following for a realistic fictional {crime_type} incident:
- A one-line dispatch message the officer would hear on the radio
- The interviewee's full name (fictional)
- The interviewee's role (victim or witness)

Use fictional names and invented addresses only.

Return ONLY a valid JSON object, no markdown fences:
{{
  "dispatch_line": "one line dispatch message",
  "interviewee_name": "fictional full name",
  "interviewee_role": "victim or witness"
}}"""

    meta_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": meta_prompt}],
        temperature=TEMP_CREATIVE,
        response_format={"type": "json_object"}
    )
    meta = parse_json_response(meta_response.choices[0].message.content)

    # Generate facts
    def generate_facts_batch(elements_batch, established_facts=None):
        elements_text = "\n".join([
            f'  {el["id"]}: {el["element"]}'
            for el in elements_batch
        ])

        context_block = ""
        if established_facts:
            context_block = f"""The following facts have already been established for this scenario.
Your answers must be consistent with them:
{json.dumps(established_facts, indent=2)}

"""

        prompt = f"""You are generating facts for a fictional {crime_type} training scenario.

        LEGAL AND PROCEDURAL RULES — these must be strictly followed:
        {legal_context}

        {crime_type_constraints}

        {SCENARIO_VARIETY_INSTRUCTIONS}

        NAMES AND ADDRESSES:
        {NAMES_AND_ADDRESSES}

        {context_block}Generate a specific concrete answer for EVERY element listed below.
        Use fictional names and invented addresses only.
        Do not use "Unknown" or vague responses — every element must have a specific answer.
        If an element asks about conduct that did not occur, provide a concrete negative fact instead of inventing that conduct.
        All facts must be internally consistent with each other, with the selected primary crime type, and with the legal rules above.
        Pay close attention to element IDs and make sure each answer corresponds exactly to its element.

        Checklist elements:
        {elements_text}

        Return ONLY a valid JSON object mapping element IDs to answers, no markdown fences:
        {{
        "element_id": "specific concrete answer",
        ...
        }}"""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP_CREATIVE,
            response_format={"type": "json_object"}
        )
        return parse_json_response(response.choices[0].message.content)

    # Batch for long checklists
    facts = {}
    if len(elements) > 40:
        mid = len(elements) // 2
        first_half = elements[:mid]
        second_half = elements[mid:]

        print(f"  Generating first batch ({len(first_half)} elements)...")
        first_facts = generate_facts_batch(first_half)
        facts.update(first_facts)

        print(f"  Generating second batch ({len(second_half)} elements)...")
        second_facts = generate_facts_batch(second_half, established_facts=first_facts)
        facts.update(second_facts)
    else:
        print(f"  Generating facts ({len(elements)} elements)...")
        facts = generate_facts_batch(elements)

    scenario = {
        "crime_type": crime_type,
        "interviewee_role": meta["interviewee_role"],
        "interviewee_name": meta["interviewee_name"],
        "dispatch_line": meta["dispatch_line"],
        "facts": facts
    }

    # Validate
    print("  Validating scenario...")
    validation_prompt = f"""You are a consistency checker for a police report training simulator.

    Review the scenario below for the crime type: {crime_type}

    LEGAL AND PROCEDURAL RULES — use these to identify and fix violations:
    {legal_context}

    {crime_type_constraints}

    {SCENARIO_VARIETY_INSTRUCTIONS}

    Check for and fix ALL of the following issues:
    - Any fact that says "unknown", "N/A", or is vague rather than specific
    - Any two facts that contradict each other
    - Any violations of the legal rules listed above
    - Any logistical inconsistencies
    - Any answers that do not correspond to their element ID
    - Any fact that turns the incident into a different primary crime type
    - Any unnecessary co-occurring crime fact that is not required by the checklist
    - Repeated, generic, placeholder, or confusingly similar fictional names or places within the same scenario
    - Scenario facts that reuse the same incident pattern unnecessarily across multiple elements instead of giving element-specific facts

    Here are the checklist elements for reference:
    {elements_text_full}

    Here is the scenario to review and correct:
    {json.dumps(scenario, indent=2)}

    Return ONLY the corrected scenario as a valid JSON object with the exact same structure,
    no markdown fences, no extra text.
    If no corrections are needed, return the scenario unchanged."""

    validation_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": validation_prompt}],
        temperature=TEMP_DETERMINISTIC,
        response_format={"type": "json_object"}
    )
    validated_scenario = parse_json_response(validation_response.choices[0].message.content)

    # Check for missing elements or unknowns
    missing = []
    unknowns = []
    for el in elements:
        if el["id"] not in validated_scenario["facts"]:
            missing.append(el["id"])
        elif validated_scenario["facts"][el["id"]].strip().lower() in ["unknown", "n/a", ""]:
            unknowns.append(el["id"])

    if missing:
        print(f"  Warning: missing facts for elements: {missing}")
    if unknowns:
        print(f"  Warning: unknowns still present after validation: {unknowns}")

    return validated_scenario